In [1]:
import torch
from transformers import pipeline

print(f"✅ PyTorch version : {torch.__version__}")
print("✅ Transformers est bien installé !")

/usr/local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch version : 2.8.0+cpu
✅ Transformers est bien installé !


In [2]:
import pandas as pd
from pymongo import MongoClient
from transformers import pipeline

print("🚀 Initialisation du Pipeline RoBERTa...")

# 1. Connexion à la Base de Données
client = MongoClient("mongodb://mongodb:27017/")
db = client['thumalien_db']
collection = db['raw_posts']

# 2. Récupération des posts (Texte + URI pour identifier)
cursor = collection.find({}, {"_id": 0, "uri": 1, "text": 1})
df = pd.DataFrame(list(cursor))
print(f"📡 Données chargées : {len(df)} posts prêts à être analysés.")

# 3. Chargement du Modèle RoBERTa (Le "Challenger")
# Nous utilisons 'hamzab/roberta-fake-news-classification' qui est une référence
# La première fois, cela va télécharger environ 500Mo (patience !)
print("🧠 Téléchargement du modèle RoBERTa (cela peut prendre 1 à 2 minutes)...")

classifier = pipeline(
    "text-classification", 
    model="hamzab/roberta-fake-news-classification", 
    tokenizer="hamzab/roberta-fake-news-classification",
    top_k=None,   # On veut tous les scores (Vrai ET Faux)
    device=-1     # -1 signifie CPU (plus stable sur Docker Mac M4 ici)
)

print("✅ Modèle RoBERTa chargé et prêt !")

🚀 Initialisation du Pipeline RoBERTa...
📡 Données chargées : 23800 posts prêts à être analysés.
🧠 Téléchargement du modèle RoBERTa (cela peut prendre 1 à 2 minutes)...


Device set to use cpu


✅ Modèle RoBERTa chargé et prêt !


In [7]:
import pandas as pd
from pymongo import MongoClient, UpdateOne
from transformers import pipeline
import time

print("🚀 Démarrage du Pipeline Expert : Zero-Shot + Fake News Detection...")

# --- 1. CHARGEMENT DES DONNÉES ---
client = MongoClient("mongodb://mongodb:27017/")
db = client['thumalien_db']
collection = db['raw_posts']

# OPTIMISATION EXPERT : On ne traite que les données fraîches pour le monitoring temps réel
# On prend les 500 derniers (ce qui prendra ~10-15 minutes max au lieu de 14h)
cursor = collection.find({}, {"_id": 0, "uri": 1, "text": 1}).sort("_id", -1).limit(500)

df = pd.DataFrame(list(cursor))
print(f"📡 Données chargées pour analyse immédiate : {len(df)} posts récents.")

# --- 2. CHARGEMENT DES MODÈLES (Double Intelligence) ---

# A. Le Trieur d'Intention (BART Zero-Shot)
# Il va dire SI on doit vérifier le tweet
print("🧠 Chargement du modèle de classification d'intention (Zero-Shot)...")
classifier_intent = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)

# B. Le Vérificateur de Fake News (RoBERTa)
# Il ne sera activé QUE pour les affirmations factuelles
print("🧠 Chargement du modèle de détection de Fake News (RoBERTa)...")
classifier_fake = pipeline("text-classification", model="hamzab/roberta-fake-news-classification", device=-1)

print("✅ Pipeline Double-Cerveau prêt.")

# --- 3. ANALYSE INTELLIGENTE ---
print("🏃‍♂️ Analyse en cours (Intention -> Vérification)...")

ops = []
batch_size = 8 # Plus petit car on a deux modèles
total = len(df)

# Labels que l'on veut que l'IA reconnaisse sans entraînement préalable
candidate_labels = ["factual news claim", "personal opinion", "commercial spam"]

start_time = time.time()

for i in range(0, total, batch_size):
    batch_df = df.iloc[i : i+batch_size]
    texts = batch_df['text'].astype(str).tolist()
    
    try:
        # ÉTAPE 1 : Quelle est l'intention ? (Zero-Shot)
        intent_results = classifier_intent(texts, candidate_labels)
        
        # On traite chaque tweet du batch
        for j, intent_res in enumerate(intent_results):
            text_analyzed = texts[j]
            uri = batch_df.iloc[j]['uri']
            
            # On récupère le label gagnant (ex: "personal opinion")
            top_intent = intent_res['labels'][0]
            intent_score = intent_res['scores'][0]
            
            # LOGIQUE EXPERTE DÉCISIONNELLE
            
            if top_intent == "personal opinion":
                # C'est subjectif ("J'aime les chats"), donc on ne juge pas la vérité.
                # On met une crédibilité haute mais on note que c'est une opinion.
                credibility = 0.95
                prediction = 0 # Fiable
                ai_log = "Opinion (Non vérifiable)"
                
            elif top_intent == "commercial spam":
                # C'est du spam ("Buy crypto now")
                credibility = 0.10
                prediction = 1 # Alerte Spam
                ai_log = "Spam détecté"
                
            elif top_intent == "factual news claim":
                # C'est une affirmation ("Le président a signé...") -> LÀ ON VÉRIFIE !
                # On appelle le 2ème modèle (RoBERTa) juste pour ce texte
                fake_check = classifier_fake(text_analyzed[:512]) # Truncation manuelle rapide
                
                # Gestion résultat liste/dict
                res_fake = fake_check[0] if isinstance(fake_check, list) else fake_check
                
                if res_fake['label'] in ['FAKE', 'Fake', 'LABEL_0']:
                    credibility = 1 - res_fake['score']
                    prediction = 1 # Vraie Alerte Fake News
                    ai_log = "Affirmation Factuelle -> JUGÉE FAUSSE"
                else:
                    credibility = res_fake['score']
                    prediction = 0
                    ai_log = "Affirmation Factuelle -> JUGÉE VRAIE"
            
            # Préparation sauvegarde
            ops.append(
                UpdateOne(
                    {"uri": uri}, 
                    {"$set": {
                        "ai_score_credibility": float(credibility),
                        "prediction_label": int(prediction),
                        "ai_intent_category": top_intent, # On garde la trace de l'intention
                        "ai_analysis_log": ai_log
                    }}
                )
            )
            
    except Exception as e:
        print(f"⚠️ Erreur batch {i}: {e}")

    # Progression
    if i % 100 == 0:
        print(f"✅ Traités : {i}/{total}")

# --- 4. SAUVEGARDE ---
if ops:
    print(f"💾 Sauvegarde de {len(ops)} analyses intelligentes...")
    collection.bulk_write(ops)
    print("🎉 TERMINÉ. Niveau Expert atteint.")

🚀 Démarrage du Pipeline Expert : Zero-Shot + Fake News Detection...
📡 Données chargées pour analyse immédiate : 500 posts récents.
🧠 Chargement du modèle de classification d'intention (Zero-Shot)...


Device set to use cpu


🧠 Chargement du modèle de détection de Fake News (RoBERTa)...


Device set to use cpu


✅ Pipeline Double-Cerveau prêt.
🏃‍♂️ Analyse en cours (Intention -> Vérification)...


...

✅ Traités : 400/500


/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffeade671f0> was reported to be 8(when accessing len(dataloader)), but 9 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffeade671f0> was reported to be 8(when accessing len(dataloader)), but 10 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffeade671f0> was reported to be 8(when accessing len(dataloader)), but 11 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:7

⚠️ Erreur batch 472: You must include at least one label and at least one sequence.


/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffdfe1bbee0> was reported to be 8(when accessing len(dataloader)), but 9 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffdfe1bbee0> was reported to be 8(when accessing len(dataloader)), but 10 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:751: UserWarning: Length of IterableDataset <transformers.pipelines.pt_utils.PipelineChunkIterator object at 0xfffdfe1bbee0> was reported to be 8(when accessing len(dataloader)), but 11 samples have been fetched. 
  warnings.warn(warn_msg)
/usr/local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:7

💾 Sauvegarde de 484 analyses intelligentes...
🎉 TERMINÉ. Niveau Expert atteint.
